# 01 — Setup & pilot data (Colab T4)

Clone **musicinstruct**, install the LLM stack, authenticate Hugging Face, generate the **300-item pilot** (if needed), validate, and self-test the grader.

Run **Sync to Drive** at the end so notebook **02** can reuse the dataset in a new session.

**Runtime:** GPU → T4 (or better).

In [ ]:
# @title Configuration
MUSICINSTRUCT_REPO = "https://github.com/juliagsy/musicinstruct.git"
BRANCH = "main"  # @param {type:"string"}

WORK = "/content"
MUSICINSTRUCT = f"{WORK}/musicinstruct"
RUN_ID = "pilot_v1"  # @param {type:"string"}  reuse in notebooks 02/03

USE_DRIVE = True  # @param {type:"boolean"}
DRIVE_ROOT = "/content/drive/MyDrive/musicinstruct"
DRIVE_DATA = f"{DRIVE_ROOT}/data/pilot"
TARGET_ITEMS = 300  # @param {type:"integer"}
FORCE_REGENERATE = False  # @param {type:"boolean"}  overwrite local + Drive pilot data

In [ ]:
# @title Mount Google Drive (optional)
from pathlib import Path

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(DRIVE_DATA).parent.mkdir(parents=True, exist_ok=True)
    print("Drive data path:", DRIVE_DATA)
else:
    print("Drive disabled — data stays on this VM only.")

In [ ]:
# @title Hugging Face token (Llama 3.2 license required)
import os
from getpass import getpass

if not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = getpass("HF token (hf_...): ")

from huggingface_hub import login

login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)

In [ ]:
# @title Clone repo & install
import shutil
from pathlib import Path

if Path(MUSICINSTRUCT).is_dir():
    shutil.rmtree(MUSICINSTRUCT)
!git clone --branch {BRANCH} --depth 1 {MUSICINSTRUCT_REPO} {MUSICINSTRUCT}
%cd {MUSICINSTRUCT}
!pip install -q -U "transformers>=4.44" accelerate huggingface_hub
!pip install -q -e ".[llm]"

In [ ]:
# @title GPU check
import torch

print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
else:
    print("Warning: no GPU — plan executor will be slow. Enable T4 in Runtime settings.")

In [ ]:
# @title Restore pilot data from Drive (or generate)
import shutil
from pathlib import Path

local_data = Path(MUSICINSTRUCT) / "data/pilot"
manifest = local_data / "pilot.jsonl"
drive_manifest = Path(DRIVE_DATA) / "pilot.jsonl" if USE_DRIVE else None

if FORCE_REGENERATE and local_data.exists():
    shutil.rmtree(local_data)
elif USE_DRIVE and drive_manifest and drive_manifest.is_file() and not manifest.is_file():
    shutil.copytree(DRIVE_DATA, local_data, dirs_exist_ok=True)
    print("Restored pilot from Drive:", drive_manifest)

if not manifest.is_file():
    force_flag = "--force" if local_data.exists() else ""
    !musicinstruct generate-pilot --output-dir data/pilot --target {TARGET_ITEMS} {force_flag}
else:
    print("Using manifest:", manifest)

In [ ]:
# @title Validate & self-test grader
!musicinstruct validate data/pilot/pilot.jsonl
!musicinstruct self-test data/pilot/pilot.jsonl

In [ ]:
# @title Dataset summary
import json
from collections import Counter
from pathlib import Path

rows = [json.loads(line) for line in Path("data/pilot/pilot.jsonl").read_text().splitlines() if line.strip()]
print("items:", len(rows))
print("splits:", dict(Counter(r["split"] for r in rows)))
print("test compositions:", sorted({r["composition_id"] for r in rows if r["split"] == "test"}))

In [ ]:
# @title Sync pilot data to Google Drive
import shutil
from pathlib import Path

local_data = Path(MUSICINSTRUCT) / "data/pilot"
if USE_DRIVE:
    Path(DRIVE_DATA).parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(local_data, DRIVE_DATA, dirs_exist_ok=True)
    print("Synced →", DRIVE_DATA)
    print("files:", len(list(Path(DRIVE_DATA).rglob("*"))))
else:
    print("Skipped Drive sync. Local path:", local_data)

## Next

1. Confirm Drive sync above.
2. Open **`02_run_plan_executor.ipynb`** (same `RUN_ID`, `USE_DRIVE=True`).